# CS781 Capstone — Phase 2: Legal Named Entity Recognition (L-NER)
## Full implementation on the Phase-1 Transformer-XL + GQA foundation model

This notebook is a **complete, runnable implementation** (not a scaffold) of all ten Phase-2
experiment slots required/allowed by the capstone brief, built on top of the Transformer-XL + GQA
model trained in Phase 1 (`best_model_transformer_xl_gqa-small.pt`, ~76M parameters).

**Compliance with the brief's "You must" list:**
- Every experiment slot below loads and starts from the **Phase-1 checkpoint**. No other pretrained
  model, pretrained weights, or external LLM API is used anywhere in this notebook.
- Each slot fine-tunes/adapts the Phase-1 backbone on the provided L-NER data and adds a
  token-classification (BIO) **or** span-extraction head on top of it.
- Ten experiment slots are implemented: **Full FT, Partial/layer-wise FT, LoRA, LORMA
  (mandatory), Houlsby/Pfeiffer Adapters, Knowledge Distillation (mandatory), Prefix Tuning,
  Prompt Tuning, BitFit, Span Extraction.** You only need 6 for the minimum requirement (5 adaptation
  techniques + 1 KD), but all ten are wired up so you can freely choose which 6+ to report.
- Training configuration (lr, batch size, window length, overlap) is exposed per-experiment so you
  can investigate it yourself — this notebook does **not** pick your final hyperparameters,
  run hyperparameter search, or write your result analysis/conclusions for you, per the
  LLM-usage rule in the brief.

**What this notebook investigates (brief §8, "You should ... investigate"):**
span-vs-token modelling (E10 vs E1–E9), document/window context size (§ "Context-window study"),
character↔token alignment (§ "Alignment"), class-balancing (weighted CE, § "Class imbalance"),
task-specific preprocessing, transfer from the Phase-1 foundation model (every slot), different
prediction heads (BIO vs span), different span-decoding strategies (greedy BIO decode vs
start/end pointer decode, plus a window-stitching/dedup strategy), and domain-specific error
analysis (boundary errors, label confusion, boilerplate false positives, long-document errors).

**LORMA note:** implemented exactly as in the cited paper — Bihany, Patel & Modi (2025),
*"LoRMA: Low-Rank Multiplicative Adaptation for LLMs"* (arXiv:2506.07621) — i.e. a genuine
multiplicative low-rank update `h = I(BA) · W₀x` with both the permutation-inflation (`LoRMA_π`)
and additive-inflation (`LoRMA_+`) variants, not a re-badged LoRA.


## 1. Environment

In [ ]:
import os, gc, json, math, time, random, re, copy, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("device:", device)


device: cpu


## 2. Paths and global configuration

Edit these paths to match your Drive/local layout.

In [ ]:
import os
from google.colab import drive
from datasets import load_from_disk, Dataset, DatasetDict

# Mount Google Drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
# ---- Phase-1 artifacts (edit to your paths) ----
VOCAB_SIZE = 32768

PHASE1_ROOT = os.environ.get(
    "PHASE1_ROOT",
    "/content/drive/MyDrive/CS781/Capstone-Project/Phase_1_Pretraining",
)
PHASE1_CHECKPOINT_DIR = os.path.join(PHASE1_ROOT, "transformer_xl_gqa_checkpoints")
PHASE1_CHECKPOINT = os.path.join(PHASE1_ROOT, "best_model_transformer_xl_gqa-small.pt")
TOKENIZER_PATH = os.path.join(PHASE1_ROOT, f"bpe_{VOCAB_SIZE}.json")          # Phase-1 BPE tokenizer
MODEL_CONFIG_PATH = os.path.join(PHASE1_ROOT, "model_config-small-txl.json")

# ---- Phase-2 data ----
PHASE2_ROOT = os.environ.get(
    "PHASE2_ROOT",
    "/content/drive/MyDrive/CS781/Capstone-Project/Phase_2_LNER",
)
TRAIN_DIR = os.path.join(PHASE2_ROOT, "Train")
TEST_DIR = os.path.join(PHASE2_ROOT, "Test")
PHASE2_OUT = os.path.join(PHASE2_ROOT, "outputs")
CKPT_DIR = os.path.join(PHASE2_OUT, "checkpoints")
os.makedirs(PHASE2_OUT, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print("PHASE1_CHECKPOINT:", PHASE1_CHECKPOINT)
print("PHASE2_ROOT:", PHASE2_ROOT)


PHASE1_CHECKPOINT: /content/drive/MyDrive/CS781/Capstone-Project/Phase_1_Pretraining/best_model_transformer_xl_gqa-small.pt
PHASE2_ROOT: /content/drive/MyDrive/CS781/Capstone-Project/Phase_2_LNER


In [ ]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()


## 3. Phase-1 Transformer-XL + GQA backbone

Copied **unmodified** from the Phase-1 notebook so `load_state_dict` matches exactly.
Do not "clean up" these classes without re-checking the checkpoint still loads.

In [ ]:
class TransformerXLRelativePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        if d_model % 2:
            raise ValueError("d_model must be even")
        self.d_model = d_model
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, query_length: int, key_length: int, device=None):
        if key_length <= 0:
            raise ValueError("key_length must be positive")
        if device is None:
            device = self.inv_freq.device
        pos_seq = torch.arange(key_length - 1, -1, -1, dtype=torch.float32, device=device)
        sinusoid_inp = torch.outer(pos_seq, self.inv_freq.to(device))
        return torch.cat([sinusoid_inp.sin(), sinusoid_inp.cos()], dim=-1)


class TransformerXLGQAAttention(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0):
        super().__init__()
        if d_model % num_query_heads != 0:
            raise ValueError("d_model must be divisible by num_query_heads")
        if num_query_heads % num_kv_heads != 0:
            raise ValueError("num_query_heads must be divisible by num_kv_heads")
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.group_size = num_query_heads // num_kv_heads

        self.q_proj = nn.Linear(d_model, num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.r_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.u = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        self.v = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        nn.init.normal_(self.u, std=0.02)
        nn.init.normal_(self.v, std=0.02)
        self.dropout = nn.Dropout(dropout)

    def _q_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_query_heads, self.head_dim).transpose(1, 2)

    def _kv_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

    def _expand_kv(self, x):
        B, H, T, D = x.shape
        return x.unsqueeze(2).expand(B, H, self.group_size, T, D).reshape(B, self.num_query_heads, T, D)

    @staticmethod
    def _rel_shift(x):
        B, H, Q, K = x.shape
        zero = torch.zeros((B, H, Q, 1), dtype=x.dtype, device=x.device)
        x = torch.cat([zero, x], dim=-1)
        x = x.view(B, H, K + 1, Q)
        x = x[:, :, 1:].view(B, H, Q, K)
        return x

    @staticmethod
    def _causal_mask(query_length, key_length, device):
        memory_length = key_length - query_length
        return torch.triu(
            torch.ones((query_length, key_length), dtype=torch.bool, device=device),
            diagonal=memory_length + 1,
        )

    def forward(self, x, memory=None, relative_position=None):
        B, Q, _ = x.shape
        if memory is not None and memory.size(1) > 0:
            kv_input = torch.cat([memory, x], dim=1)
        else:
            kv_input = x
        K = kv_input.size(1)

        q = self._q_heads(self.q_proj(x))
        k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
        v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))

        ac = torch.einsum("bhqd, bhkd->bhqk", q, k)
        ac = ac + torch.einsum("hd,bhkd->bhk", self.u, k).unsqueeze(2)

        if relative_position is None:
            raise ValueError("relative_position is required for Transformer-XL attention")

        r = self._expand_kv(self._kv_heads(self.r_proj(relative_position).unsqueeze(0)))
        bd = torch.einsum("bhqd,bhkd->bhqk", q, r)
        bd = bd + torch.einsum("hd,bhkd->bhk", self.v, r).unsqueeze(2)
        bd = self._rel_shift(bd)

        scores = (ac + bd) / math.sqrt(self.head_dim)
        mask = self._causal_mask(Q, K, x.device)
        scores = scores.masked_fill(mask.view(1, 1, Q, K), torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, Q, self.d_model)
        return self.out_proj(out)


class TransformerXLBlockGQA(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0, ffn_mult=4):
        super().__init__()
        self.attn = TransformerXLGQAAttention(d_model, num_query_heads, num_kv_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        ffn_dim = ffn_mult * d_model
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, memory, relative_position):
        x = self.norm1(x + self.attn(x, memory=memory, relative_position=relative_position))
        x = self.norm2(x + self.ffn(x))
        return x


class TransformerXLGQAModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_query_heads, num_kv_heads,
                 memory_length, dropout=0.0, tie_embeddings=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.memory_length = memory_length
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = TransformerXLRelativePositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerXLBlockGQA(d_model, num_query_heads, num_kv_heads, dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.embedding.weight

    def _update_memory(self, old_memory, x):
        if self.memory_length <= 0:
            return None
        x = x.detach()
        if old_memory is None:
            return x[:, -self.memory_length:].contiguous()
        return torch.cat([old_memory, x], dim=1)[:, -self.memory_length:].detach().contiguous()

    def forward(self, input_ids, memory=None, labels=None):
        x = self.embedding(input_ids)
        Q = x.size(1)
        if memory is None:
            memory = [None] * self.num_layers
        if len(memory) != self.num_layers:
            raise ValueError(f"memory must contain {self.num_layers} entries")
        mem_len = 0 if memory[0] is None else memory[0].size(1)
        K = mem_len + Q
        rel_pos = self.pos_encoding(Q, K, device=x.device)
        new_memory = []
        for i, block in enumerate(self.blocks):
            x = block(x, memory[i], rel_pos)
            new_memory.append(self._update_memory(memory[i], x))
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), labels.reshape(-1))
        return {"logits": logits, "loss": loss, "memory": new_memory}


def save_transformer_xl_checkpoint(filepath, model, optimizer=None, scheduler=None, scaler=None,
                                    epoch=0, global_step=0, best_val_loss=float("inf"), loss_history=None):
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    torch.save({
        "epoch": epoch, "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "scaler_state_dict": scaler.state_dict() if scaler and scaler.is_enabled() else None,
        "best_val_loss": best_val_loss,
        "loss_history": loss_history or [],
    }, filepath)


def load_transformer_xl_checkpoint(filepath, model, optimizer=None, scheduler=None, scaler=None, device="cpu"):
    if not filepath or not os.path.exists(filepath):
        return 0, 0, float("inf"), []
    ckpt = torch.load(filepath, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    if optimizer is not None and ckpt.get("optimizer_state_dict") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler is not None and scaler.is_enabled() and ckpt.get("scaler_state_dict") is not None:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    return ckpt.get("epoch", 0), ckpt.get("global_step", 0), ckpt.get("best_val_loss", float("inf")), ckpt.get("loss_history", [])

print("Backbone classes defined.")


Backbone classes defined.


## 4. Load the Phase-1 tokenizer and checkpoint

In [ ]:
assert os.path.exists(TOKENIZER_PATH), (
    "Tokenizer not found. Point TOKENIZER_PATH to the BPE JSON saved in Phase 1."
)
tokenizer = BPETokenizer.from_json(TOKENIZER_PATH)   # BPETokenizer class is defined in your Phase-1 notebook
vocab_size = len(tokenizer.itos)
print("vocab_size =", vocab_size)

if os.path.exists(MODEL_CONFIG_PATH):
    with open(MODEL_CONFIG_PATH, "r") as f:
        phase1_cfg = json.load(f)
else:
    # Fallback: the exact Phase-1 "small" config used to train best_model_transformer_xl_gqa-small.pt
    phase1_cfg = {
        "vocab_size": vocab_size, "d_model": 768, "num_query_heads": 32,
        "num_kv_heads": 8, "memory_length": 128, "dropout": 0.1,
    }
NUM_LAYERS = 8  # fixed in Phase-1 training config
print("phase1_cfg:", phase1_cfg)

def build_phase1_backbone():
    """Fresh, Phase-1-initialized backbone (weights loaded from PHASE1_CHECKPOINT)."""
    m = TransformerXLGQAModel(
        vocab_size=phase1_cfg["vocab_size"],
        d_model=phase1_cfg["d_model"],
        num_layers=NUM_LAYERS,
        num_query_heads=phase1_cfg["num_query_heads"],
        num_kv_heads=phase1_cfg["num_kv_heads"],
        memory_length=phase1_cfg["memory_length"],
        dropout=phase1_cfg.get("dropout", 0.1),
    )
    assert os.path.exists(PHASE1_CHECKPOINT), "Phase-1 checkpoint not found at PHASE1_CHECKPOINT."
    _, _, best_val_loss, _ = load_transformer_xl_checkpoint(PHASE1_CHECKPOINT, m, device="cpu")
    print("Loaded Phase-1 checkpoint. best_val_loss =", best_val_loss,
          "| params =", sum(p.numel() for p in m.parameters()))
    return m.to(device)

phase1_model = build_phase1_backbone()
phase1_model.eval()
D_MODEL = phase1_cfg["d_model"]


vocab_size = 32768
phase1_cfg: {'vocab_size': 32768, 'd_model': 768, 'num_query_heads': 32, 'num_kv_heads': 8, 'memory_length': 128, 'dropout': 0.1}
Loaded Phase-1 checkpoint. best_val_loss = 3.077281383833674 | params = 75959808


## 5. Phase-2 L-NER data

`fold1`/`fold2` are the two public training folds; `fold3` is the private test fold.
We use fold1 as the training split and fold2 as our **local dev split** for model selection
(the brief provides no separate dev split for Phase 2 — §8.3.2).

In [ ]:
from huggingface_hub import hf_hub_download, login

# Uncomment if your environment needs an explicit HF auth step (private repo / rate limits):
# login()

PHASE2_REPO_ID = "Exploration-Lab/CS-781-Capstone"

login()

def hf_phase2_path(relative_path):
    """relative_path is relative to Phase_2_LNER/, e.g. 'Train/fold1-text.json'."""
    return hf_hub_download(
        repo_id=PHASE2_REPO_ID,
        repo_type="dataset",
        filename=f"Phase_2_LNER/{relative_path}",
    )

PHASE2_FILES = {
    "fold1_text": hf_phase2_path("Train/fold1-text.json"),
    "fold1_gold": hf_phase2_path("Train/fold1-gold.json"),
    "fold2_text": hf_phase2_path("Train/fold2-text.json"),
    "fold2_gold": hf_phase2_path("Train/fold2-gold.json"),
    "labels":     hf_phase2_path("Train/labels.txt"),
    "fold3_text": hf_phase2_path("Test/fold3-text.json"),
}

for name, path in PHASE2_FILES.items():
    print(f"{name:12s} -> {path}")


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

fold1_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json
fold1_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json
fold2_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json
fold2_gold   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json
labels       -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt
fold3_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb

Inspect the raw schema once before trusting any downstream assumption about it.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

with open(PHASE2_FILES["labels"], "r", encoding="utf-8") as f:
    LABELS = [line.strip() for line in f if line.strip()]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}

fold1_text, fold1_gold = load_json(PHASE2_FILES["fold1_text"]), load_json(PHASE2_FILES["fold1_gold"])
fold2_text, fold2_gold = load_json(PHASE2_FILES["fold2_text"]), load_json(PHASE2_FILES["fold2_gold"])

print("labels:", LABELS)
print("num labels:", len(LABELS))
print("fold1 (train) documents:", len(fold1_text))
print("fold2 (dev)   documents:", len(fold2_text))


labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
num labels: 12
fold1 (train) documents: 35
fold2 (dev)   documents: 35


In [ ]:
for name, obj in [("fold1_text", fold1_text), ("fold1_gold", fold1_gold)]:
    doc_id, val = next(iter(obj.items()))
    print(name, "example id:", doc_id)
    print(repr(val)[:600])
    print()


fold1_text example id: 115651329
'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.0

fold1_gold example id: 115651329
[[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana'], [359, 369, 'COURT', 'High Court'], [575, 585, 'DATE', '18.11.1994'], [637, 644, 'WIT', 'Jagdish'], [695, 704, 'WIT', 'Mange Ram'], [792, 800, 'APP', 'Jage Ram'], [1122, 1130, 'APP', 'Jage Ram'], [1341,

## 6. Character ↔ token alignment

The Phase-2 brief is explicit that an entity "correct only when its start offset, end offset, and
label exactly match the reference annotation" (strict span matching, §8.3.5/§8.4). Any drift
between the custom BPE tokenizer's pieces and the raw character string is fatal, so alignment is
built and **audited** (§15, "Alignment audit") before any training happens.

Approach: regex-tokenize into words/punctuation first (so each BPE piece is searched for only
inside its own word's character window), rather than searching the whole document per piece —
this avoids silently matching the wrong occurrence of a common sub-word.

In [ ]:
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def token_spans_from_custom_bpe(text, tokenizer):
    """Returns (token_ids, char_spans) with one (start,end) per token id.
    Special tokens (e.g. <s>) get span (-1, -1) and are excluded from span decoding."""
    token_ids = tokenizer.encode(text, add_special_tokens=True)
    pieces = [tokenizer.itos[i] for i in token_ids]

    spans = [None] * len(token_ids)
    cursor_by_word = {}

    word_token_idx = 0
    non_special_ids = tokenizer.encode(text, add_special_tokens=False)
    # Map: leading special tokens (e.g. <s>) get (-1,-1); we detect them by count difference.
    n_leading_special = len(token_ids) - len(non_special_ids)
    for i in range(n_leading_special):
        spans[i] = (-1, -1)

    tid_ptr = n_leading_special
    for match in WORD_RE.finditer(text):
        word = match.group(0)
        word_start, word_end = match.start(), match.end()
        local_ids = tokenizer.encode(word, add_special_tokens=False)
        local_cursor = word_start
        for tid in local_ids:
            piece = tokenizer.itos[tid]
            clean = piece.replace(getattr(tokenizer, "end_marker", ""), "")
            if clean == "":
                spans[tid_ptr] = (local_cursor, local_cursor)
            else:
                pos = text.find(clean, local_cursor, word_end + 1)
                if pos < 0:
                    pos = local_cursor  # fallback: never crash alignment, just degrade gracefully
                    span_end = pos
                else:
                    span_end = pos + len(clean)
                spans[tid_ptr] = (pos, span_end)
                local_cursor = span_end
            tid_ptr += 1

    for i in range(tid_ptr, len(token_ids)):
        spans[i] = (-1, -1)  # trailing special tokens
    return token_ids, spans


### Gold-annotation normalization + BIO tag construction

In [ ]:
def normalize_gold_entities(raw):
    """Accepts the official [start,end,label] / [start,end,label,text] list form, or a dict
    wrapper, and returns a clean list of (start, end, label) tuples."""
    if raw is None:
        return []
    if isinstance(raw, list):
        out = []
        for x in raw:
            if isinstance(x, dict):
                start = x.get("start", x.get("start_character", x.get("begin")))
                end = x.get("end", x.get("end_character", x.get("stop")))
                label = x.get("label", x.get("entity_label", x.get("type")))
                if start is not None and end is not None and label is not None:
                    out.append((int(start), int(end), str(label)))
            elif isinstance(x, (list, tuple)) and len(x) >= 3:
                out.append((int(x[0]), int(x[1]), str(x[2])))
        return out
    if isinstance(raw, dict):
        for key in ["entities", "annotations", "gold", "labels"]:
            if key in raw:
                return normalize_gold_entities(raw[key])
    raise TypeError(f"Unsupported annotation schema: {type(raw)}")


BIO_LABELS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
BIO2ID = {t: i for i, t in enumerate(BIO_LABELS)}
ID2BIO = {i: t for t, i in BIO2ID.items()}


def entities_to_bio(token_spans, entities):
    """Token-level BIO tag ids. Tokens with span (-1,-1) (special tokens) get -100 (ignored)."""
    tags = []
    for (s, e) in token_spans:
        if s < 0:
            tags.append(-100)
            continue
        tag = "O"
        for (gs, ge, label) in entities:
            if s >= gs and e <= ge and e > s:
                tag = ("B-" if s == gs else "I-") + label
                break
        tags.append(BIO2ID[tag])
    return tags


def bio_ids_to_entities(pred_ids, token_spans):
    """Greedy left-to-right BIO decode -> list of (start, end, label) char spans."""
    entities = []
    current = None
    for tid, (s, e) in zip(pred_ids, token_spans):
        if s < 0:
            continue
        tag = ID2BIO[int(tid)]
        if tag == "O":
            if current is not None:
                entities.append(tuple(current)); current = None
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B":
            if current is not None:
                entities.append(tuple(current))
            current = [s, e, label]
        else:  # "I"
            if current is None or current[2] != label:
                if current is not None:
                    entities.append(tuple(current))
                current = [s, e, label]   # dangling I- treated as a new span start (lenient decode)
            else:
                current[1] = e
    if current is not None:
        entities.append(tuple(current))
    return entities


### Alignment audit (brief §15: "Alignment audit")

Checks every gold entity's character slice really reproduces the annotated text, and reports how
many entities fail to land on a clean token boundary (which strict evaluation will always mark
wrong no matter how good the model is).

In [ ]:
def audit_gold_alignment(text_dict, gold_dict, n_docs=50):
    rows = []
    boundary_misses = 0
    total = 0
    for doc_id, text in list(text_dict.items())[:n_docs]:
        ents = normalize_gold_entities(gold_dict.get(doc_id, []))
        token_ids, spans = token_spans_from_custom_bpe(text, tokenizer)
        valid_token_starts = {s for (s, e) in spans if s >= 0}
        valid_token_ends = {e for (s, e) in spans if s >= 0}
        for (s, e, label) in ents:
            total += 1
            on_boundary = (s in valid_token_starts) and (e in valid_token_ends)
            if not on_boundary:
                boundary_misses += 1
            rows.append({
                "doc_id": doc_id, "start": s, "end": e, "label": label,
                "entity_text": text[s:e], "valid_bounds": 0 <= s < e <= len(text),
                "on_token_boundary": on_boundary,
            })
    df = pd.DataFrame(rows)
    print(f"Checked {total} gold entities across {n_docs} docs.")
    print(f"Entities NOT landing cleanly on a token boundary: {boundary_misses} "
          f"({100*boundary_misses/max(total,1):.1f}%) -- these are unrecoverable under strict "
          f"BIO decoding regardless of model quality; this motivates also trying the span-extraction "
          f"head (E10), which predicts raw character offsets directly.")
    return df

alignment_report = audit_gold_alignment(fold1_text, fold1_gold, n_docs=50)
display(alignment_report.head(20))


Checked 4120 gold entities across 50 docs.
Entities NOT landing cleanly on a token boundary: 28 (0.7%) -- these are unrecoverable under strict BIO decoding regardless of model quality; this motivates also trying the span-extraction head (E10), which predicts raw character offsets directly.


,doc_id,start,end,label,entity_text,valid_bounds,on_token_boundary
0,115651329,137,153,RESP,STATE OF HARYANA,True,True
1,115651329,252,261,DATE,19.8.2011,True,True
2,115651329,276,308,COURT,High Court of Punjab and Haryana,True,True
3,115651329,359,369,COURT,High Court,True,True
4,115651329,575,585,DATE,18.11.1994,True,True
5,115651329,637,644,WIT,Jagdish,True,True
6,115651329,695,704,WIT,Mange Ram,True,True
7,115651329,792,800,APP,Jage Ram,True,True
8,115651329,1122,1130,APP,Jage Ram,True,True
9,115651329,1341,1349,APP,Jage Ram,True,True


## 7. Class imbalance audit (brief: "Class-balancing strategies", "Class imbalance")

Legal NER is heavily skewed (e.g. DATE/CASENO are common, WIT/AUTH are rare). We compute
per-label frequency and an inverse-frequency class-weight vector for the BIO head's loss.

In [ ]:
def compute_entity_counts(text_dict, gold_dict):
    counts = Counter()
    for doc_id in text_dict:
        for (s, e, label) in normalize_gold_entities(gold_dict.get(doc_id, [])):
            counts[label] += 1
    return counts

entity_counts = compute_entity_counts(fold1_text, fold1_gold)
display(pd.DataFrame(sorted(entity_counts.items(), key=lambda x: -x[1]), columns=["label", "count"]))

def bio_class_weights(entity_counts, labels=LABELS, bio_labels=BIO_LABELS, smoothing=1.0):
    """Inverse-frequency weights per BIO tag id; O gets the (down-weighted) mean weight so it
    doesn't dominate the loss just because it's the majority class."""
    weights = torch.ones(len(bio_labels))
    total = sum(entity_counts.values()) + smoothing * len(labels)
    for label in labels:
        freq = entity_counts.get(label, 0) + smoothing
        w = total / (len(labels) * freq)
        weights[BIO2ID[f"B-{label}"]] = w
        weights[BIO2ID[f"I-{label}"]] = w
    weights[BIO2ID["O"]] = weights[1:].mean() * 0.1  # keep O's weight small relative to entity tags
    return weights

CLASS_WEIGHTS = bio_class_weights(entity_counts).to(device)
print("class weight range:", CLASS_WEIGHTS.min().item(), "-", CLASS_WEIGHTS.max().item())


,label,count
0,DATE,813
1,PREC,624
2,AUTH,535
3,STAT,525
4,COURT,455
5,CASENO,423
6,APP,207
7,RESP,191
8,JUDGE,123
9,R.COUNSEL,87


class weight range: 0.19908593595027924 - 5.063725471496582


## 8. Document & sentence context windows (brief: "Document and sentence context windows")

Legal documents routinely exceed any practical sequence length. We split each document into
overlapping token windows; the overlap region lets an entity that would otherwise be cut in half
by one window appear whole in the neighbouring window. `MAX_TOKENS`/`WINDOW_OVERLAP` are exposed
as the knobs for the context-window study in §13.

In [ ]:
MAX_TOKENS = 512          # context window size (tokens) -- vary this to study context-window effects
WINDOW_OVERLAP = 64        # overlap between consecutive windows, in tokens

def prepare_document(doc_id, text, raw_gold=None):
    token_ids, spans = token_spans_from_custom_bpe(text, tokenizer)
    entities = normalize_gold_entities(raw_gold) if raw_gold is not None else []
    bio = entities_to_bio(spans, entities)
    return {"id": doc_id, "text": text, "input_ids": token_ids, "token_spans": spans,
            "entities": entities, "bio": bio}


def build_records(text_dict, gold_dict=None, max_docs=None):
    records = []
    items = list(text_dict.items())
    if max_docs is not None:
        items = items[:max_docs]
    for doc_id, text in items:
        gold = gold_dict.get(doc_id, []) if gold_dict is not None else []
        records.append(prepare_document(doc_id, text, gold))
    return records


def make_windows(record, max_tokens=MAX_TOKENS, overlap=WINDOW_OVERLAP):
    ids, spans, bio = record["input_ids"], record["token_spans"], record["bio"]
    n = len(ids)
    if n <= max_tokens:
        return [{"doc_id": record["id"], "input_ids": ids, "token_spans": spans,
                  "bio": bio, "text": record["text"]}]
    stride = max(max_tokens - overlap, 1)
    windows = []
    start = 0
    while start < n:
        end = min(start + max_tokens, n)
        windows.append({
            "doc_id": record["id"],
            "input_ids": ids[start:end],
            "token_spans": spans[start:end],
            "bio": bio[start:end],
            "text": record["text"],
        })
        if end == n:
            break
        start += stride
    return windows


print("Building windowed records (train=fold1, dev=fold2)...")
train_records = build_records(fold1_text, fold1_gold)
dev_records = build_records(fold2_text, fold2_gold)

train_windows = [w for r in train_records for w in make_windows(r)]
dev_windows = [w for r in dev_records for w in make_windows(r)]
print(f"train docs={len(train_records)} -> windows={len(train_windows)}")
print(f"dev   docs={len(dev_records)} -> windows={len(dev_windows)}")

gold_by_doc_dev = {doc_id: normalize_gold_entities(fold2_gold.get(doc_id, [])) for doc_id in fold2_text}


Building windowed records (train=fold1, dev=fold2)...
train docs=35 -> windows=543
dev   docs=35 -> windows=553


## 9. Datasets and collators (BIO head and span head)

In [ ]:
PAD_ID = tokenizer.stoi.get("<pad>", 0)

class NERWindowDataset(Dataset):
    """Token-classification (BIO) formulation."""
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        w = self.windows[i]
        return {
            "doc_id": w["doc_id"],
            "input_ids": torch.tensor(w["input_ids"], dtype=torch.long),
            "labels": torch.tensor(w["bio"], dtype=torch.long),
            "token_spans": w["token_spans"],
        }

    def __getitems__(self, indices):
        return [self.__getitem__(idx) for idx in indices]


def ner_collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["input_ids"])
        input_ids[i, :n] = x["input_ids"]
        labels[i, :n] = x["labels"]
    return {
        "input_ids": input_ids, "labels": labels,
        "doc_id": [x["doc_id"] for x in batch],
        "token_spans": [x["token_spans"] for x in batch],
    }


def entities_to_span_tags(token_spans, entities, label2id):
    """Per-token start/end tag ids for the span-extraction head.
    Tag id = label index for the first/last token of an entity of that label, else len(labels) ("none")."""
    none_id = len(label2id)
    start_tags = [none_id] * len(token_spans)
    end_tags = [none_id] * len(token_spans)
    span_to_idx = {}
    for idx, (s, e) in enumerate(token_spans):
        if s >= 0:
            span_to_idx.setdefault(s, idx)  # first token index whose span starts at s
    for (gs, ge, label) in entities:
        if label not in label2id:
            continue
        start_idx = end_idx = None
        for idx, (s, e) in enumerate(token_spans):
            if s < 0:
                continue
            if s == gs and start_idx is None:
                start_idx = idx
            if e == ge:
                end_idx = idx
        if start_idx is not None:
            start_tags[start_idx] = label2id[label]
        if end_idx is not None:
            end_tags[end_idx] = label2id[label]
    return start_tags, end_tags


class SpanWindowDataset(Dataset):
    """Start/end pointer-network formulation (brief: "span extraction versus token-aligned")."""
    def __init__(self, windows, label2id=LABEL2ID):
        self.windows = windows
        self.label2id = label2id

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        w = self.windows[i]
        # Recover char-span entities that fall inside this window from the window's own bio tags
        # by re-deriving from doc-level entities filtered to this window's token span range.
        start_tags, end_tags = entities_to_span_tags(w["token_spans"], w.get("entities_in_window", []), self.label2id)
        return {
            "doc_id": w["doc_id"],
            "input_ids": torch.tensor(w["input_ids"], dtype=torch.long),
            "start_tags": torch.tensor(start_tags, dtype=torch.long),
            "end_tags": torch.tensor(end_tags, dtype=torch.long),
            "token_spans": w["token_spans"],
        }

    def __getitems__(self, indices):
        return [self.__getitem__(idx) for idx in indices]


def span_collate(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    none_id = len(LABELS)
    input_ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    start_tags = torch.full((len(batch), max_len), -100, dtype=torch.long)
    end_tags = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, x in enumerate(batch):
        n = len(x["input_ids"])
        input_ids[i, :n] = x["input_ids"]
        start_tags[i, :n] = x["start_tags"]
        end_tags[i, :n] = x["end_tags"]
    return {
        "input_ids": input_ids, "start_tags": start_tags, "end_tags": end_tags,
        "doc_id": [x["doc_id"] for x in batch],
        "token_spans": [x["token_spans"] for x in batch],
    }


def attach_entities_in_window(windows, doc_entities_by_id):
    """For the span dataset: keep only gold entities whose char span is fully inside this window's
    token-span range (avoids training on truncated entities at window edges)."""
    out = []
    for w in windows:
        valid_spans = [s for (s, e) in w["token_spans"] if s >= 0]
        if not valid_spans:
            w = dict(w); w["entities_in_window"] = []
            out.append(w); continue
        lo, hi = min(valid_spans), max(e for (s, e) in w["token_spans"] if s >= 0)
        ents = [(s, e, l) for (s, e, l) in doc_entities_by_id.get(w["doc_id"], []) if s >= lo and e <= hi]
        w = dict(w); w["entities_in_window"] = ents
        out.append(w)
    return out

doc_entities_train = {doc_id: normalize_gold_entities(fold1_gold.get(doc_id, [])) for doc_id in fold1_text}
doc_entities_dev = {doc_id: normalize_gold_entities(fold2_gold.get(doc_id, [])) for doc_id in fold2_text}
train_windows_span = attach_entities_in_window(train_windows, doc_entities_train)
dev_windows_span = attach_entities_in_window(dev_windows, doc_entities_dev)
print("Span-formulation windows ready.")

Span-formulation windows ready.


## 10. Strict span evaluator (primary metric: Strict Macro-F1, brief §8.4)

A predicted entity counts only if `(doc_id, start, end, label)` exactly matches a gold entity —
"a prediction that identifies the correct entity but has an incorrect character boundary is
considered incorrect" (brief §8.4).

In [ ]:
def strict_span_metrics(gold_by_doc, pred_by_doc, labels=LABELS):
    per_label = {}
    for label in labels:
        gold = {(d, int(s), int(e)) for d, ents in gold_by_doc.items() for (s, e, l) in ents if l == label}
        pred = {(d, int(s), int(e)) for d, ents in pred_by_doc.items() for (s, e, l) in ents if l == label}
        tp = len(gold & pred); fp = len(pred - gold); fn = len(gold - pred)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[label] = {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}
    return {
        "macro_f1": float(np.mean([v["f1"] for v in per_label.values()])),
        "macro_precision": float(np.mean([v["precision"] for v in per_label.values()])),
        "macro_recall": float(np.mean([v["recall"] for v in per_label.values()])),
        "per_label": per_label,
    }


def dedupe_window_predictions(doc_predictions, strategy="union"):
    """Span-decoding / window-stitching strategy (brief: 'different span-decoding strategies').
    'union': keep every distinct (start,end,label) seen in any window covering that region.
    Overlapping windows can occasionally emit near-duplicate or conflicting spans for the same
    text at different offsets if windows were not aligned on char boundaries; since our windows
    are token-aligned and spans are always derived from token_spans, duplicates are exact and a
    simple set union is sufficient and lossless."""
    final = {}
    for doc_id, ents in doc_predictions.items():
        if strategy == "union":
            final[doc_id] = sorted(set((int(s), int(e), str(l)) for s, e, l in ents),
                                    key=lambda x: (x[0], x[1], x[2]))
        else:
            raise ValueError(f"unknown dedupe strategy: {strategy}")
    return final


## 11. Backbone feature extraction and prediction heads

`transformer_xl_features` runs the backbone over a window **without** carrying Transformer-XL
recurrent memory across windows (each window is classified independently — appropriate for
NER, where we don't want information about one window's content to silently leak into another's
boundary decisions through persistent memory). Prefix tuning (E7) overrides this by feeding
trainable memory instead of `None` — see §12.6.

In [ ]:
def transformer_xl_features(model, input_ids, memory=None):
    x = model.embedding(input_ids)
    Q = x.size(1)
    if memory is None:
        memory = [None] * model.num_layers
    mem_len = 0 if memory[0] is None else memory[0].size(1)
    K = mem_len + Q
    rel_pos = model.pos_encoding(Q, K, device=x.device)
    new_memory = []
    for i, block in enumerate(model.blocks):
        x = block(x, memory[i], rel_pos)
        new_memory.append(model._update_memory(memory[i], x))
    x = model.final_norm(x)
    return x, new_memory


class BioNERHead(nn.Module):
    """E1-E9: token classification over the BIO tag space."""
    def __init__(self, d_model, num_bio_labels):
        super().__init__()
        self.classifier = nn.Linear(d_model, num_bio_labels)

    def forward(self, hidden):
        return self.classifier(hidden)


class SpanNERHead(nn.Module):
    """E10: independent start/end pointer classifiers per token, each over
    (num_entity_labels + 1 'none') classes. See §12 'Span-decoding strategies' for how
    start/end logits are paired back into character spans."""
    def __init__(self, d_model, num_entity_labels):
        super().__init__()
        self.num_entity_labels = num_entity_labels
        self.start_proj = nn.Linear(d_model, num_entity_labels + 1)
        self.end_proj = nn.Linear(d_model, num_entity_labels + 1)

    def forward(self, hidden):
        return self.start_proj(hidden), self.end_proj(hidden)


def decode_span_head(start_logits, end_logits, token_spans, none_id, max_span_tokens=40):
    """Greedy nearest-start/end pairing per label (pointer-network decode)."""
    start_pred = start_logits.argmax(-1).tolist()
    end_pred = end_logits.argmax(-1).tolist()
    n = len(token_spans)
    starts_by_label = defaultdict(list)
    ends_by_label = defaultdict(list)
    for i in range(n):
        s_char, e_char = token_spans[i]
        if s_char < 0:
            continue
        if start_pred[i] != none_id:
            starts_by_label[start_pred[i]].append(i)
        if end_pred[i] != none_id:
            ends_by_label[end_pred[i]].append(i)

    entities = []
    for label_id in set(list(starts_by_label.keys()) + list(ends_by_label.keys())):
        used_ends = set()
        for si in starts_by_label.get(label_id, []):
            best_ei = None
            for ei in ends_by_label.get(label_id, []):
                if ei >= si and ei - si <= max_span_tokens and ei not in used_ends:
                    if best_ei is None or ei < best_ei:
                        best_ei = ei
            if best_ei is not None:
                used_ends.add(best_ei)
                s_char = token_spans[si][0]
                e_char = token_spans[best_ei][1]
                entities.append((s_char, e_char, LABELS[label_id]))
    return entities


## 12. Fine-tuning and parameter-efficient adaptation techniques

Each technique below is a **real, trainable implementation**, not a placeholder. They're injected
into a *fresh copy* of the Phase-1 backbone (`copy.deepcopy(phase1_model)`), so experiments never
interfere with each other's weights.

| id | technique | where it's applied |
|---|---|---|
| 12.1 | Full fine-tuning | everything |
| 12.2 | Partial / layer-wise fine-tuning | top-`k` blocks + final_norm + head |
| 12.3 | LoRA | `q_proj`, `v_proj` of every attention block |
| 12.4 | **LORMA** (mandatory) | `q_proj`, `v_proj`, `out_proj` of every attention block |
| 12.5 | Houlsby / Pfeiffer Adapters | after attention and/or FFN sub-layers |
| 12.6 | Prefix tuning | trainable per-layer memory (reuses the XL memory slot) |
| 12.7 | Prompt tuning | trainable virtual tokens prepended to the input |
| 12.8 | BitFit | bias terms only (+ head) |
| 12.9 | Knowledge distillation | trains a smaller **student** backbone from a trained teacher |


### 12.1 / 12.2 — Full fine-tuning and partial / layer-wise fine-tuning

In [ ]:
def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {"trainable": trainable, "total": total, "trainable_pct": 100.0 * trainable / max(total, 1)}


def apply_full_finetuning(backbone):
    for p in backbone.parameters():
        p.requires_grad = True
    return backbone


def apply_partial_layerwise(backbone, num_trainable_top_layers=2):
    """Freeze the embedding and the bottom (num_layers - k) blocks; only the top-k blocks,
    final_norm, and (always) the task head are trainable."""
    for p in backbone.parameters():
        p.requires_grad = False
    n = backbone.num_layers
    for i, block in enumerate(backbone.blocks):
        if i >= n - num_trainable_top_layers:
            for p in block.parameters():
                p.requires_grad = True
    for p in backbone.final_norm.parameters():
        p.requires_grad = True
    return backbone


### 12.3 — LoRA (Hu et al., 2021)

Additive low-rank update: `h = W0 x + (alpha/r) B A x`, `B` zero-init, `A` is random-init.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        out_f, in_f = base_linear.out_features, base_linear.in_features
        self.r, self.alpha = r, alpha
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_f, r))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        base_out = self.base(x)
        delta = self.dropout(x) @ self.A.t() @ self.B.t()
        return base_out + (self.alpha / self.r) * delta


def _iter_target_linears(backbone, target_names=("q_proj", "v_proj")):
    """Yield (attn_module, attr_name) for every attention projection matching target_names."""
    for block in backbone.blocks:
        for name in target_names:
            yield block.attn, name


def apply_lora(backbone, r=8, alpha=16, dropout=0.0, target_names=("q_proj", "v_proj")):
    for p in backbone.parameters():
        p.requires_grad = False
    for attn, name in _iter_target_linears(backbone, target_names):
        base = getattr(attn, name)
        setattr(attn, name, LoRALinear(base, r=r, alpha=alpha, dropout=dropout))
    return backbone


### 12.4 — LORMA (mandatory): Low-Rank **Multiplicative** Adaptation
Bihany, Patel & Modi (2025), arXiv:2506.07621. Implements the actual paper, not LoRA-with-a-new-name:

`h = I(BA) · W0 x`, i.e. the update **multiplies** the frozen layer's output rather than adding to
its weight. Naive `BA` caps the output at rank `r`, so the paper introduces two rank-inflation
operators `I`:

- **LoRMA₊** (additive inflation): `I₊(BA) = (α/r)·BA + I_d`, so `h = W0x + (α/r)·BA·(W0x)`.
  Init `B=0`, `A~N(0,0.02²)` ⇒ starts at the identity transform.
- **LoRMA_π** (permutation inflation): rotate row *i* of `BA` cyclically by *i* positions, which
  provably restores (near) full rank. Init: first column of `B`=1, rest random; `A[0,0]=1`, rest 0
  ⇒ `I_π(BA) = I_d` at init.

Both are provided; `variant=` selects which one an experiment uses.

In [ ]:
def _permutation_inflate(M):
    """Cyclically roll row i of a (..., d, d) matrix by i positions along the last dim."""
    d = M.size(-1)
    idx = torch.arange(d, device=M.device)
    shifted_idx = (idx.unsqueeze(0) - idx.unsqueeze(1)) % d   # [d, d]
    return torch.gather(M, -1, shifted_idx.unsqueeze(0).expand(M.size(0), -1, -1))


class LoRMALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r=8, alpha=8, variant="plus"):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        out_f = base_linear.out_features
        self.r, self.alpha, self.variant = r, alpha, variant
        self.B = nn.Parameter(torch.empty(out_f, r))
        self.A = nn.Parameter(torch.empty(r, out_f))
        if variant == "plus":
            # I_+(BA) = (alpha/r) BA + I_d  -> identity at init requires B = 0
            nn.init.zeros_(self.B)
            nn.init.normal_(self.A, std=0.02)
        elif variant == "pi":
            # I_pi(BA) = I_d at init: first column of B = 1, A[0,0] = 1, rest 0
            nn.init.normal_(self.B, std=0.02)
            with torch.no_grad():
                self.B[:, 0] = 1.0
            nn.init.zeros_(self.A)
            with torch.no_grad():
                self.A[0, 0] = 1.0
        else:
            raise ValueError("variant must be 'plus' or 'pi'")

    def forward(self, x):
        y0 = self.base(x)                      # W0 x,  shape [..., out_f]
        BA = self.B @ self.A                    # [out_f, out_f]
        flat = y0.reshape(-1, y0.size(-1)).unsqueeze(0)  # [1, N, out_f] batch dim for permute helper
        if self.variant == "plus":
            delta = F.linear(y0, BA)             # y0 @ BA^T  == (BA) y0 in matrix-vector form
            out = y0 + (self.alpha / self.r) * delta
        else:  # "pi"
            BA_inflated = _permutation_inflate(BA.unsqueeze(0)).squeeze(0)
            out = F.linear(y0, BA_inflated)
        return out


def apply_lorma(backbone, r=8, alpha=8, variant="plus", target_names=("q_proj", "v_proj", "out_proj")):
    for p in backbone.parameters():
        p.requires_grad = False
    for attn, name in _iter_target_linears(backbone, target_names):
        base = getattr(attn, name)
        setattr(attn, name, LoRMALinear(base, r=r, alpha=alpha, variant=variant))
    return backbone


### 12.5 — Houlsby / Pfeiffer Adapters
Bottleneck adapter: `down -> nonlinearity -> up`, up-projection zero-init so the adapter starts as
a no-op (identity residual). **Houlsby**: one adapter after attention, one after the FFN (two
insertion points per block). **Pfeiffer**: a single adapter after the FFN only.

In [ ]:
class Adapter(nn.Module):
    def __init__(self, d_model, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


class AdapterBlock(nn.Module):
    """Wraps an existing TransformerXLBlockGQA, inserting adapter(s) without touching its
    original forward/weights."""
    def __init__(self, block, d_model, bottleneck=64, variant="houlsby"):
        super().__init__()
        self.block = block
        self.variant = variant
        self.adapter_ffn = Adapter(d_model, bottleneck)
        self.adapter_attn = Adapter(d_model, bottleneck) if variant == "houlsby" else None

    def forward(self, x, memory, relative_position):
        attn_out = self.block.attn(x, memory=memory, relative_position=relative_position)
        if self.adapter_attn is not None:
            attn_out = self.adapter_attn(attn_out)
        x = self.block.norm1(x + attn_out)
        ffn_out = self.block.ffn(x)
        ffn_out = self.adapter_ffn(ffn_out)
        x = self.block.norm2(x + ffn_out)
        return x


def apply_adapters(backbone, bottleneck=64, variant="houlsby"):
    for p in backbone.parameters():
        p.requires_grad = False
    for i in range(len(backbone.blocks)):
        backbone.blocks[i] = AdapterBlock(backbone.blocks[i], backbone.d_model, bottleneck, variant)
    return backbone


### 12.6 — Prefix tuning
Reuses the Transformer-XL **memory** slot that the backbone already supports: instead of real
recurrent memory, every layer receives a trainable "virtual" key/value context of length
`prefix_len`, prepended before the real window. The backbone itself is fully frozen.

In [ ]:
class PrefixTunedBackbone(nn.Module):
    def __init__(self, backbone, prefix_len=16):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.prefix_len = prefix_len
        self.prefix_embeds = nn.Parameter(
            torch.randn(backbone.num_layers, prefix_len, backbone.d_model) * 0.02
        )

    def features(self, input_ids):
        b = self.backbone
        x = b.embedding(input_ids)
        B, Q, _ = x.shape
        K = self.prefix_len + Q
        rel_pos = b.pos_encoding(Q, K, device=x.device)
        for i, block in enumerate(b.blocks):
            mem_i = self.prefix_embeds[i].unsqueeze(0).expand(B, -1, -1)
            x = block(x, mem_i, rel_pos)
        x = b.final_norm(x)
        return x

    @property
    def num_layers(self):
        return self.backbone.num_layers

    @property
    def d_model(self):
        return self.backbone.d_model


def apply_prefix_tuning(backbone, prefix_len=16):
    return PrefixTunedBackbone(backbone, prefix_len=prefix_len)


### 12.7 — Prompt tuning
`prompt_len` trainable virtual **token embeddings** are prepended to the real embedded sequence
before the first block; the backbone is frozen and processes `[prompt; real tokens]` as one
sequence. The prompt positions are dropped again before the NER head sees the hidden states, since
they don't correspond to real characters.

In [ ]:
class PromptTunedBackbone(nn.Module):
    def __init__(self, backbone, prompt_len=16):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.prompt_len = prompt_len
        self.prompt_embeds = nn.Parameter(
            torch.randn(prompt_len, backbone.d_model) * 0.02
        )

    def features(self, input_ids):
        b = self.backbone
        x = b.embedding(input_ids)
        B, Q, _ = x.shape
        prompt = self.prompt_embeds.unsqueeze(0).expand(B, -1, -1)
        x = torch.cat([prompt, x], dim=1)
        Qp = x.size(1)
        rel_pos = b.pos_encoding(Qp, Qp, device=x.device)
        for block in b.blocks:
            x = block(x, None, rel_pos)
        x = b.final_norm(x)
        return x[:, self.prompt_len:, :]   # drop virtual-token positions

    @property
    def num_layers(self):
        return self.backbone.num_layers

    @property
    def d_model(self):
        return self.backbone.d_model


def apply_prompt_tuning(backbone, prompt_len=16):
    return PromptTunedBackbone(backbone, prompt_len=prompt_len)


### 12.8 — BitFit (Ben Zaken et al., 2022)
Only bias parameters are trainable. Note this backbone uses `bias=False` on every attention
projection and the LM head, so in practice BitFit here unfreezes the **FFN linear biases** and
**LayerNorm biases** — which is exactly the intended "tiny parameter budget" regime of BitFit.

In [ ]:
def apply_bitfit(backbone):
    for name, p in backbone.named_parameters():
        p.requires_grad = name.endswith(".bias")
    return backbone


## 13. Unified NER model wrapper

Works uniformly over a plain backbone (full-FT / partial-FT / LoRA / LORMA / Adapters / BitFit)
and the prefix/prompt-tuned wrapper backbones, and over both heads (BIO / span).

In [ ]:
def backbone_features(backbone, input_ids):
    if hasattr(backbone, "features"):          # PrefixTunedBackbone / PromptTunedBackbone
        return backbone.features(input_ids)
    hidden, _ = transformer_xl_features(backbone, input_ids, memory=None)
    return hidden


class NERModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, input_ids):
        hidden = backbone_features(self.backbone, input_ids)
        return self.head(hidden)


### 12.9 — Knowledge distillation (mandatory)

Per the brief: *"use a larger Phase-1/Phase-2 model as the teacher and train a smaller student
model to transfer useful task-specific knowledge."* Here the **teacher** is the fully fine-tuned
Phase-1-backbone NER model (E1, ~76M backbone parameters). The **student** is a genuinely smaller
Transformer-XL+GQA backbone (fewer layers, smaller `d_model`) trained from scratch purely from the
distillation signal (soft teacher logits + hard gold labels) — it never loads any external
pretrained weights, so this stays within the "no other pretrained model/weights" rule while still
producing the "compact version" the brief's Phase-1 section (§7.4) anticipates.

In [ ]:
def build_student_backbone(d_model=256, num_layers=4, num_query_heads=16, num_kv_heads=4,
                            memory_length=64, dropout=0.1):
    """A small, RANDOMLY INITIALIZED Transformer-XL+GQA backbone -- the distillation student.
    Not loaded from any pretrained checkpoint; it learns purely from the KD objective below."""
    student = TransformerXLGQAModel(
        vocab_size=vocab_size, d_model=d_model, num_layers=num_layers,
        num_query_heads=num_query_heads, num_kv_heads=num_kv_heads,
        memory_length=memory_length, dropout=dropout,
    ).to(device)
    for p in student.parameters():
        p.requires_grad = True
    return student


def distillation_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5, ignore_index=-100):
    """Standard Hinton et al. (2015) KD objective:
    alpha * KL(student_soft || teacher_soft) * T^2  +  (1-alpha) * CE(student, hard_labels)."""
    mask = labels.ne(ignore_index)
    ce = F.cross_entropy(
        student_logits.reshape(-1, student_logits.size(-1)),
        labels.reshape(-1), ignore_index=ignore_index,
    )
    T = temperature
    student_log_probs = F.log_softmax(student_logits / T, dim=-1)
    teacher_probs = F.softmax(teacher_logits.detach() / T, dim=-1)
    kd_per_token = F.kl_div(student_log_probs, teacher_probs, reduction="none").sum(-1)  # [B,T]
    kd = (kd_per_token * mask).sum() / mask.sum().clamp(min=1)
    kd = kd * (T * T)
    return alpha * kd + (1.0 - alpha) * ce


## 14. Training loops (BIO, span, and KD variants)

In [ ]:
def train_ner_bio(model, loader, epochs, lr, weight_decay=0.0, class_weights=None,
                   grad_clip=1.0, max_batches=None, log_every=50):
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(epochs):
        model.train()
        running, count = 0.0, 0
        for bidx, batch in enumerate(loader):
            if max_batches is not None and bidx >= max_batches:
                break
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(input_ids)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                ignore_index=-100, weight=class_weights,
            )
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], grad_clip)
            optimizer.step()
            running += float(loss.item()); count += 1
            if (bidx + 1) % log_every == 0:
                print(f"  epoch={epoch+1} batch={bidx+1}/{len(loader)} loss={running/count:.4f}")
        epoch_loss = running / max(count, 1)
        history.append(epoch_loss)
        print(f"epoch={epoch+1} mean_loss={epoch_loss:.5f}")
    return history


def train_ner_span(model, loader, epochs, lr, weight_decay=0.0, grad_clip=1.0,
                    max_batches=None, log_every=50):
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)
    history = []
    for epoch in range(epochs):
        model.train()
        running, count = 0.0, 0
        for bidx, batch in enumerate(loader):
            if max_batches is not None and bidx >= max_batches:
                break
            input_ids = batch["input_ids"].to(device)
            start_tags = batch["start_tags"].to(device)
            end_tags = batch["end_tags"].to(device)
            optimizer.zero_grad(set_to_none=True)
            start_logits, end_logits = model(input_ids)
            loss = (
                F.cross_entropy(start_logits.reshape(-1, start_logits.size(-1)), start_tags.reshape(-1), ignore_index=-100)
                + F.cross_entropy(end_logits.reshape(-1, end_logits.size(-1)), end_tags.reshape(-1), ignore_index=-100)
            )
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], grad_clip)
            optimizer.step()
            running += float(loss.item()); count += 1
            if (bidx + 1) % log_every == 0:
                print(f"  epoch={epoch+1} batch={bidx+1}/{len(loader)} loss={running/count:.4f}")
        epoch_loss = running / max(count, 1)
        history.append(epoch_loss)
        print(f"epoch={epoch+1} mean_loss={epoch_loss:.5f}")
    return history


def train_ner_kd(student_model, teacher_model, loader, epochs, lr, temperature=2.0, alpha=0.5,
                  grad_clip=1.0, max_batches=None, log_every=50):
    teacher_model.eval()
    for p in teacher_model.parameters():
        p.requires_grad = False
    optimizer = AdamW([p for p in student_model.parameters() if p.requires_grad], lr=lr)
    history = []
    for epoch in range(epochs):
        student_model.train()
        running, count = 0.0, 0
        for bidx, batch in enumerate(loader):
            if max_batches is not None and bidx >= max_batches:
                break
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            with torch.no_grad():
                teacher_logits = teacher_model(input_ids)
            optimizer.zero_grad(set_to_none=True)
            student_logits = student_model(input_ids)
            loss = distillation_loss(student_logits, teacher_logits, labels, temperature=temperature, alpha=alpha)
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(student_model.parameters(), grad_clip)
            optimizer.step()
            running += float(loss.item()); count += 1
            if (bidx + 1) % log_every == 0:
                print(f"  [KD] epoch={epoch+1} batch={bidx+1}/{len(loader)} loss={running/count:.4f}")
        epoch_loss = running / max(count, 1)
        history.append(epoch_loss)
        print(f"[KD] epoch={epoch+1} mean_loss={epoch_loss:.5f}")
    return history


## 15. Prediction

In [ ]:
@torch.no_grad()
def predict_windows_bio(model, windows, batch_size=4):
    loader = DataLoader(NERWindowDataset(windows), batch_size=batch_size, shuffle=False, collate_fn=ner_collate)
    model.eval()
    doc_predictions = defaultdict(list)
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        logits = model(input_ids)
        pred = logits.argmax(-1).cpu().numpy()
        for i in range(len(pred)):
            ents = bio_ids_to_entities(pred[i], batch["token_spans"][i])
            doc_predictions[batch["doc_id"][i]].extend(ents)
    return dedupe_window_predictions(doc_predictions)


@torch.no_grad()
def predict_windows_span(model, windows, batch_size=4):
    loader = DataLoader(SpanWindowDataset(windows), batch_size=batch_size, shuffle=False, collate_fn=span_collate)
    model.eval()
    none_id = len(LABELS)
    doc_predictions = defaultdict(list)
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        start_logits, end_logits = model(input_ids)
        for i in range(input_ids.size(0)):
            ents = decode_span_head(start_logits[i], end_logits[i], batch["token_spans"][i], none_id)
            doc_predictions[batch["doc_id"][i]].extend(ents)
    return dedupe_window_predictions(doc_predictions)


## 16. Experiment registry

Flip `"enabled"` per slot. `E1` and the mandatory `E4` (LORMA) and `E6` (KD) are on by default;
turn on any other 3+ to reach the "5 different adaptation techniques" minimum. Changing only a
listed hyperparameter (`lr`, `rank`, `bottleneck`, window length, ...) within one technique does
**not** count as a new technique under the brief's rules — it's for your own tuning, not for
padding the experiment count.

In [ ]:
EXPERIMENTS = {
    "E1_full_ft":   {"method": "full_ft",        "head": "bio",  "enabled": False,  "epochs": 3, "lr": 2e-5},
    "E2_partial_ft":{"method": "partial_ft",     "head": "bio",  "enabled": True, "epochs": 4, "lr": 5e-5,
                      "kwargs": {"num_trainable_top_layers": 2}},
    "E3_lora":      {"method": "lora",           "head": "bio",  "enabled": True, "epochs": 5, "lr": 3e-4,
                      "kwargs": {"r": 8, "alpha": 16}},
    "E4_lorma":     {"method": "lorma",          "head": "bio",  "enabled": False,  "epochs": 5, "lr": 3e-4,
                      "kwargs": {"r": 8, "alpha": 8, "variant": "plus"}, "MANDATORY": True},
    "E5_adapter":   {"method": "adapter",        "head": "bio",  "enabled": True, "epochs": 5, "lr": 1e-4,
                      "kwargs": {"bottleneck": 64, "variant": "houlsby"}},
    "E6_kd":        {"method": "kd",             "head": "bio",  "enabled": False,  "epochs": 5, "lr": 1e-3,
                      "kwargs": {"temperature": 2.0, "alpha": 0.5}, "MANDATORY": True},
    "E7_prefix":    {"method": "prefix",         "head": "bio",  "enabled": False, "epochs": 5, "lr": 3e-4,
                      "kwargs": {"prefix_len": 16}},
    "E8_prompt":    {"method": "prompt",         "head": "bio",  "enabled": False, "epochs": 5, "lr": 3e-4,
                      "kwargs": {"prompt_len": 16}},
    "E9_bitfit":    {"method": "bitfit",         "head": "bio",  "enabled": True, "epochs": 5, "lr": 5e-4},
    "E10_span":     {"method": "lorma",          "head": "span", "enabled": False, "epochs": 5, "lr": 3e-4,
                      "kwargs": {"r": 8, "alpha": 8, "variant": "plus"}},
}
display(pd.DataFrame(EXPERIMENTS).T)


,method,head,enabled,epochs,lr,kwargs,MANDATORY
E1_full_ft,full_ft,bio,False,3,0.00002,NaN,NaN
E2_partial_ft,partial_ft,bio,True,4,0.00005,{'num_trainable_top_layers': 2},NaN
E3_lora,lora,bio,True,5,0.0003,"{'r': 8, 'alpha': 16}",NaN
E4_lorma,lorma,bio,False,5,0.0003,"{'r': 8, 'alpha': 8, 'variant': 'plus'}",True
E5_adapter,adapter,bio,True,5,0.0001,"{'bottleneck': 64, 'variant': 'houlsby'}",NaN
E6_kd,kd,bio,False,5,0.001,"{'temperature': 2.0, 'alpha': 0.5}",True
E7_prefix,prefix,bio,False,5,0.0003,{'prefix_len': 16},NaN
E8_prompt,prompt,bio,False,5,0.0003,{'prompt_len': 16},NaN
E9_bitfit,bitfit,bio,True,5,0.0005,NaN,NaN
E10_span,lorma,span,False,5,0.0003,"{'r': 8, 'alpha': 8, 'variant': 'plus'}",NaN


### Builder: method name -> (trainable model, backbone_kind)

In [ ]:
def build_experiment_backbone(method, kwargs=None):
    kwargs = kwargs or {}
    backbone = copy.deepcopy(phase1_model)
    if method == "full_ft":
        return apply_full_finetuning(backbone)
    if method == "partial_ft":
        return apply_partial_layerwise(backbone, **kwargs)
    if method == "lora":
        return apply_lora(backbone, **kwargs)
    if method == "lorma":
        return apply_lorma(backbone, **kwargs)
    if method == "adapter":
        return apply_adapters(backbone, **kwargs)
    if method == "prefix":
        return apply_prefix_tuning(backbone, **kwargs)
    if method == "prompt":
        return apply_prompt_tuning(backbone, **kwargs)
    if method == "bitfit":
        return apply_bitfit(backbone)
    raise ValueError(f"Unknown method: {method}")


def build_head(head_kind, d_model):
    if head_kind == "bio":
        return BioNERHead(d_model, len(BIO_LABELS))
    if head_kind == "span":
        return SpanNERHead(d_model, len(LABELS))
    raise ValueError(f"Unknown head: {head_kind}")


## 17. Experiment runner

Records not just the final F1 but exactly what the brief asks experiments to be compared on
(§8, "not only ... final F1 score but also ... exact span boundaries, entity labels,
computational efficiency, and model complexity"): trainable/total parameter counts, peak GPU
memory, wall-clock train time, and the full per-label precision/recall/F1 breakdown.

In [ ]:
RESULTS_PATH = os.path.join(PHASE2_OUT, "phase2_experiment_results.csv")
RESULT_COLUMNS = [
    "experiment", "method", "head", "window_tokens", "overlap",
    "strict_macro_f1", "strict_macro_precision", "strict_macro_recall",
    "trainable_params", "total_params", "trainable_pct",
    "peak_gpu_gb", "train_seconds", "notes",
]
_trained_models = {}   # experiment_name -> trained NERModel (kept in memory for error analysis / KD teacher)


def record_result(row):
    df = pd.DataFrame([row])
    if os.path.exists(RESULTS_PATH):
        old = pd.read_csv(RESULTS_PATH)
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(RESULTS_PATH, index=False)
    return df


def run_experiment(name, cfg, train_windows=train_windows, dev_windows=dev_windows,
                    train_windows_span=train_windows_span, dev_windows_span=dev_windows_span,
                    batch_size=4, teacher_name="E1_full_ft"):
    print("\n" + "=" * 70)
    print(f"Running {name}  (method={cfg['method']}, head={cfg['head']})")
    print("=" * 70)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    kwargs = cfg.get("kwargs", {})

    if cfg["method"] == "kd":
        assert teacher_name in _trained_models, (
            f"KD needs a trained teacher -- run '{teacher_name}' first (or point teacher_name at "
            f"any already-trained experiment in _trained_models)."
        )
        teacher_model = _trained_models[teacher_name]
        student_backbone = build_student_backbone()
        student_head = build_head(cfg["head"], student_backbone.d_model)
        model = NERModel(student_backbone, student_head).to(device)
        loader = DataLoader(NERWindowDataset(train_windows), batch_size=batch_size, shuffle=True, collate_fn=ner_collate)
        train_ner_kd(model, teacher_model, loader, epochs=cfg["epochs"], lr=cfg["lr"], **kwargs)
        pred_by_doc = predict_windows_bio(model, dev_windows, batch_size=batch_size)

    elif cfg["head"] == "span":
        backbone = build_experiment_backbone(cfg["method"], kwargs)
        head = build_head("span", backbone.d_model)
        model = NERModel(backbone, head).to(device)
        for p in model.head.parameters():
            p.requires_grad = True
        loader = DataLoader(SpanWindowDataset(train_windows_span), batch_size=batch_size, shuffle=True, collate_fn=span_collate)
        train_ner_span(model, loader, epochs=cfg["epochs"], lr=cfg["lr"])
        pred_by_doc = predict_windows_span(model, dev_windows_span, batch_size=batch_size)

    else:
        backbone = build_experiment_backbone(cfg["method"], kwargs)
        head = build_head("bio", backbone.d_model)
        model = NERModel(backbone, head).to(device)
        for p in model.head.parameters():
            p.requires_grad = True
        loader = DataLoader(NERWindowDataset(train_windows), batch_size=batch_size, shuffle=True, collate_fn=ner_collate)
        use_weights = CLASS_WEIGHTS if cfg["method"] != "bitfit" else None  # keep BitFit's tiny budget simple
        train_ner_bio(model, loader, epochs=cfg["epochs"], lr=cfg["lr"], class_weights=use_weights)
        pred_by_doc = predict_windows_bio(model, dev_windows, batch_size=batch_size)

    metrics = strict_span_metrics(gold_by_doc_dev, pred_by_doc)
    params = count_trainable(model)
    train_seconds = time.time() - t0
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else float("nan")

    print(f"strict_macro_f1={metrics['macro_f1']:.4f} | "
          f"trainable={params['trainable']:,} ({params['trainable_pct']:.2f}%) | "
          f"time={train_seconds/60:.1f} min | peak_gpu={peak_gb:.2f} GB")

    row = {
        "experiment": name, "method": cfg["method"], "head": cfg["head"],
        "window_tokens": MAX_TOKENS, "overlap": WINDOW_OVERLAP,
        "strict_macro_f1": metrics["macro_f1"],
        "strict_macro_precision": metrics["macro_precision"],
        "strict_macro_recall": metrics["macro_recall"],
        "trainable_params": params["trainable"], "total_params": params["total"],
        "trainable_pct": params["trainable_pct"], "peak_gpu_gb": peak_gb,
        "train_seconds": train_seconds, "notes": "",
    }
    record_result(row)
    torch.save({"model_state_dict": model.state_dict(), "config": cfg},
               os.path.join(CKPT_DIR, f"{name}.pt"))
    _trained_models[name] = model
    return model, metrics, pred_by_doc


### Run every enabled experiment

In [ ]:
all_metrics = {}
all_predictions = {}
for name, cfg in EXPERIMENTS.items():
    if not cfg.get("enabled", False):
        continue
    model, metrics, pred_by_doc = run_experiment(name, cfg)
    all_metrics[name] = metrics
    all_predictions[name] = pred_by_doc
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if os.path.exists(RESULTS_PATH):
    display(pd.read_csv(RESULTS_PATH))



Running E2_partial_ft  (method=partial_ft, head=bio)
  epoch=1 batch=50/136 loss=1.4932
  epoch=1 batch=100/136 loss=1.2010
epoch=1 mean_loss=1.05597
  epoch=2 batch=50/136 loss=0.5394
  epoch=2 batch=100/136 loss=0.5347
epoch=2 mean_loss=0.50471
  epoch=3 batch=50/136 loss=0.4037
  epoch=3 batch=100/136 loss=0.3994
epoch=3 mean_loss=0.37222
  epoch=4 batch=50/136 loss=0.3137
  epoch=4 batch=100/136 loss=0.2956
epoch=4 mean_loss=0.29663
strict_macro_f1=0.1877 | trainable=12,718,873 (16.74%) | time=219.9 min | peak_gpu=nan GB

Running E3_lora  (method=lora, head=bio)
  epoch=1 batch=50/136 loss=1.6451
  epoch=1 batch=100/136 loss=1.3875
epoch=1 mean_loss=1.23003
  epoch=2 batch=50/136 loss=0.5997
  epoch=2 batch=100/136 loss=0.6199
epoch=2 mean_loss=0.60581
  epoch=3 batch=50/136 loss=0.4808
  epoch=3 batch=100/136 loss=0.4715
epoch=3 mean_loss=0.45037
  epoch=4 batch=50/136 loss=0.4151
  epoch=4 batch=100/136 loss=0.3882
epoch=4 mean_loss=0.37664
  epoch=5 batch=50/136 loss=0.3700
  e

,experiment,method,head,window_tokens,overlap,strict_macro_f1,strict_macro_precision,strict_macro_recall,trainable_params,total_params,trainable_pct,peak_gpu_gb,train_seconds,notes
0,E1_full_ft,full_ft,bio,512,64,0.253221,0.215382,0.329792,75979033,75979033,100.000000,NaN,15887.687239,NaN
1,E4_lorma,lorma,bio,512,64,0.234768,0.191900,0.335829,240409,76200217,0.315496,NaN,22389.949690,NaN
2,E6_kd,kd,bio,512,64,0.188243,0.189753,0.204362,11224857,11224857,100.000000,NaN,10903.568855,NaN
3,E1_full_ft,full_ft,bio,512,64,0.226497,0.185821,0.314376,75979033,75979033,100.000000,NaN,15412.246446,NaN
4,E2_partial_ft,partial_ft,bio,512,64,0.187725,0.158456,0.256574,12718873,75979033,16.739977,NaN,13192.829149,NaN
5,E3_lora,lora,bio,512,64,0.197587,0.160817,0.277746,178969,76138777,0.235056,NaN,20433.181749,NaN
6,E5_adapter,adapter,bio,512,64,0.179222,0.144863,0.253415,1605401,77565209,2.069744,NaN,20256.743297,NaN
7,E9_bitfit,bitfit,bio,512,64,0.072648,0.071736,0.080674,63001,75979033,0.082919,NaN,19697.282084,NaN


## 18. Document / sentence context-window study (brief: "Document and sentence context windows")

Retrains the cheapest configuration (LORMA, since it's mandatory anyway) at a couple of different
`(max_tokens, overlap)` settings and compares dev strict Macro-F1, so you can see how much
window size actually matters for this dataset before spending a full budget of experiments on it.
Extend `WINDOW_SETTINGS` to cover whatever range you want to study.

In [ ]:
WINDOW_SETTINGS = [
    {"max_tokens": 256, "overlap": 32},
    {"max_tokens": 512, "overlap": 64},
    # {"max_tokens": 768, "overlap": 96},  # uncomment if your GPU budget allows it
]

def run_context_window_study(settings=WINDOW_SETTINGS, method="lorma", epochs=2, lr=3e-4, batch_size=4):
    rows = []
    for setting in settings:
        mt, ov = setting["max_tokens"], setting["overlap"]
        print(f"\n--- window study: max_tokens={mt}, overlap={ov} ---")
        tw = [w for r in train_records for w in make_windows(r, max_tokens=mt, overlap=ov)]
        dw = [w for r in dev_records for w in make_windows(r, max_tokens=mt, overlap=ov)]

        backbone = build_experiment_backbone(method, EXPERIMENTS["E4_lorma"]["kwargs"])
        head = build_head("bio", backbone.d_model)
        model = NERModel(backbone, head).to(device)
        for p in model.head.parameters():
            p.requires_grad = True
        loader = DataLoader(NERWindowDataset(tw), batch_size=batch_size, shuffle=True, collate_fn=ner_collate)
        train_ner_bio(model, loader, epochs=epochs, lr=lr, class_weights=CLASS_WEIGHTS)
        pred_by_doc = predict_windows_bio(model, dw, batch_size=batch_size)
        metrics = strict_span_metrics(gold_by_doc_dev, pred_by_doc)
        rows.append({"max_tokens": mt, "overlap": ov, "num_windows_train": len(tw),
                     "num_windows_dev": len(dw), "strict_macro_f1": metrics["macro_f1"]})
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.DataFrame(rows)

# window_study_df = run_context_window_study()   # uncomment to run (adds real training time)
# display(window_study_df)


## 19. Domain-specific error analysis

Covers every bullet the brief lists under §8 "pay particular attention to": entity boundary
errors, confusion between semantically similar legal entity types, false positives from legal
boilerplate, and errors caused by long documents. Run this on whichever experiment's predictions
you want to inspect (`all_predictions[name]`).

In [ ]:
def iou(a, b):
    s = max(a[0], b[0]); e = min(a[1], b[1])
    inter = max(0, e - s)
    union = (a[1] - a[0]) + (b[1] - b[0]) - inter
    return inter / union if union > 0 else 0.0


def boundary_error_report(gold_by_doc, pred_by_doc, iou_threshold=0.3):
    """Predicted spans that overlap a gold span of the SAME label but don't match exactly --
    i.e. the model found the right entity but got the boundary wrong."""
    rows = []
    for doc_id, preds in pred_by_doc.items():
        golds = gold_by_doc.get(doc_id, [])
        gold_exact = {(s, e, l) for (s, e, l) in golds}
        for (ps, pe, pl) in preds:
            if (ps, pe, pl) in gold_exact:
                continue
            best = max(((iou((ps, pe), (gs, ge)), gs, ge) for (gs, ge, gl) in golds if gl == pl),
                       default=(0.0, None, None))
            if best[0] >= iou_threshold:
                rows.append({"doc_id": doc_id, "label": pl, "pred_span": (ps, pe),
                             "nearest_gold_span": (best[1], best[2]), "iou": best[0]})
    return pd.DataFrame(rows)


def label_confusion_report(gold_by_doc, pred_by_doc, iou_threshold=0.5):
    """Predicted spans that overlap a gold span (any label) with a DIFFERENT label --
    i.e. boundary roughly right but the model confused two entity types."""
    confusion = Counter()
    for doc_id, preds in pred_by_doc.items():
        golds = gold_by_doc.get(doc_id, [])
        for (ps, pe, pl) in preds:
            best_label, best_iou = None, 0.0
            for (gs, ge, gl) in golds:
                ov = iou((ps, pe), (gs, ge))
                if ov > best_iou:
                    best_iou, best_label = ov, gl
            if best_label is not None and best_iou >= iou_threshold and best_label != pl:
                confusion[(best_label, pl)] += 1   # (gold_label, predicted_label)
    df = pd.DataFrame([{"gold_label": g, "predicted_label": p, "count": c} for (g, p), c in confusion.items()])
    return df.sort_values("count", ascending=False) if len(df) else df


def false_positive_boilerplate(gold_by_doc, pred_by_doc, text_by_doc, top_n=20):
    """Most frequent predicted-but-not-gold entity text -- a proxy for boilerplate FPs
    (recurring legal phrases the model mistakes for real entities)."""
    fp_text_counts = Counter()
    for doc_id, preds in pred_by_doc.items():
        gold_exact = {(s, e, l) for (s, e, l) in gold_by_doc.get(doc_id, [])}
        text = text_by_doc.get(doc_id, "")
        for (ps, pe, pl) in preds:
            if (ps, pe, pl) not in gold_exact:
                snippet = text[ps:pe].strip().lower()
                if snippet:
                    fp_text_counts[(snippet, pl)] += 1
    rows = [{"text": t, "label": l, "count": c} for (t, l), c in fp_text_counts.most_common(top_n)]
    return pd.DataFrame(rows)


def length_bucket_f1(records, gold_by_doc, pred_by_doc, buckets=((0, 2000), (2000, 8000), (8000, float("inf")))):
    """Strict Macro-F1 broken down by document character-length bucket -- isolates whether
    long documents (more windows, more context fragmentation) hurt performance."""
    doc_lengths = {r["id"]: len(r["text"]) for r in records}
    rows = []
    for lo, hi in buckets:
        doc_ids = [d for d, n in doc_lengths.items() if lo <= n < hi]
        g = {d: gold_by_doc.get(d, []) for d in doc_ids}
        p = {d: pred_by_doc.get(d, []) for d in doc_ids}
        if not doc_ids:
            continue
        m = strict_span_metrics(g, p)
        rows.append({"length_bucket": f"[{lo},{hi})", "num_docs": len(doc_ids), "strict_macro_f1": m["macro_f1"]})
    return pd.DataFrame(rows)


# Example usage once at least one experiment has run:
# name = "E1_full_ft"
# text_by_doc_dev = {r["id"]: r["text"] for r in dev_records}
# display(boundary_error_report(gold_by_doc_dev, all_predictions[name]).head(20))
# display(label_confusion_report(gold_by_doc_dev, all_predictions[name]))
# display(false_positive_boilerplate(gold_by_doc_dev, all_predictions[name], text_by_doc_dev))
# display(length_bucket_f1(dev_records, gold_by_doc_dev, all_predictions[name]))


## 20. Test-set prediction and `submission.zip` (brief §6.3, §6.4, §19)

Pick your best experiment by dev strict Macro-F1, generate predictions for every official
`fold3` test document, and package them exactly as Codabench expects: `prediction.json` at the
**root** of `submission.zip` (no wrapper folder), every official test ID present, no unknown IDs,
no duplicates.

In [ ]:
test_text = load_json(PHASE2_FILES["fold3_text"])
test_records = build_records(test_text)
test_windows = [w for r in test_records for w in make_windows(r)]
test_windows_span = attach_entities_in_window(test_windows, {})  # no gold at test time
print("test documents:", len(test_records), "| test windows:", len(test_windows))


def write_prediction_json(pred_by_doc, path, official_ids):
    missing = set(official_ids) - set(pred_by_doc.keys())
    extra = set(pred_by_doc.keys()) - set(official_ids)
    if missing:
        print(f"WARNING: {len(missing)} official IDs have no prediction -- filling with empty list.")
    if extra:
        print(f"WARNING: {len(extra)} predicted IDs are not official test IDs -- dropping them.")
    clean = {}
    for doc_id in official_ids:
        ents = pred_by_doc.get(doc_id, [])
        # Strict validation: ensure end > start
        valid_ents = []
        for (s, e, label) in ents:
            s_val, e_val = int(s), int(e)
            if e_val > s_val:
                valid_ents.append([s_val, e_val, str(label)])
            else:
                print(f"Filtered out invalid span: [{s_val}, {e_val}, {label}] in document {doc_id}")
        clean[doc_id] = valid_ents
    with open(path, "w", encoding="utf-8") as f:
        json.dump(clean, f, ensure_ascii=False, indent=2)
    print(f"Wrote {path} with {len(clean)} documents.")
    return clean


def package_submission(prediction_json_path, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(prediction_json_path, arcname="prediction.json")   # root-level, no folder wrapper
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    assert names == ["prediction.json"], f"submission.zip must contain only prediction.json at root, got: {names}"
    print(f"Packaged {zip_path} -- contents: {names}")


# ---- Load your best experiment from checkpoint and generate the official submission ----
BEST_EXPERIMENT = "E5_adapter"
cfg = EXPERIMENTS[BEST_EXPERIMENT]

# Build a fresh model structure matching the best experiment configuration
backbone = build_experiment_backbone(cfg["method"], cfg.get("kwargs", {}))
head = build_head(cfg["head"], backbone.d_model)
best_model = NERModel(backbone, head).to(device)

# Load state dict from checkpoint
checkpoint_path = os.path.join(CKPT_DIR, f"{BEST_EXPERIMENT}.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()
print(f"Successfully loaded best model weights from {checkpoint_path}")

if cfg["head"] == "span":
    test_pred = predict_windows_span(best_model, test_windows_span, batch_size=4)
else:
    test_pred = predict_windows_bio(best_model, test_windows, batch_size=4)

prediction_json_path = os.path.join(PHASE2_OUT, "prediction.json")
write_prediction_json(test_pred, prediction_json_path, official_ids=list(test_text.keys()))
package_submission(prediction_json_path, os.path.join(PHASE2_OUT, "submission.zip"))

test documents: 35 | test windows: 588
Successfully loaded best model weights from /content/drive/MyDrive/CS781/Capstone-Project/Phase_2_LNER/outputs/checkpoints/E5_adapter.pt
Filtered out invalid span: [8633, 8633, STAT] in document 91090158
Filtered out invalid span: [45356, 45356, A.COUNSEL] in document 189745935
Filtered out invalid span: [45356, 45356, R.COUNSEL] in document 189745935
Filtered out invalid span: [13151, 13151, JUDGE] in document 101482639
Filtered out invalid span: [13243, 13243, JUDGE] in document 101482639
Filtered out invalid span: [13526, 13526, CASENO] in document 101482639
Filtered out invalid span: [13530, 13530, JUDGE] in document 101482639
Filtered out invalid span: [5990, 5990, STAT] in document 10104667
Filtered out invalid span: [14095, 14095, JUDGE] in document 10104667
Wrote /content/drive/MyDrive/CS781/Capstone-Project/Phase_2_LNER/outputs/prediction.json with 35 documents.
Packaged /content/drive/MyDrive/CS781/Capstone-Project/Phase_2_LNER/outputs/s

## 21. Submission checklist (brief §19, §8.1–§8.5)

- [ ] Every experiment started from the Phase-1 checkpoint (no other pretrained weights/LLM API).
- [ ] At least 6 substantially different experiments run; at least 5 different adaptation
      techniques among them (not just hyperparameter variants of one technique).
- [ ] LORMA experiment completed (E4).
- [ ] Knowledge-distillation experiment completed (E6), using a trained teacher from this notebook.
- [ ] Character/token alignment audited (§6) — note the on-boundary miss-rate in your report.
- [ ] Strict Macro-F1 (+ per-label precision/recall) recorded for every experiment
      (`phase2_experiment_results.csv`).
- [ ] Context-window study run for at least two window sizes (§18).
- [ ] Class-imbalance strategy applied and discussed (§7 weighted CE).
- [ ] Error analysis run on your best model: boundary errors, label confusion, boilerplate false
      positives, length-bucket F1 (§19).
- [ ] Best checkpoint per serious experiment saved under `CKPT_DIR`.
- [ ] `prediction.json` contains every official `fold3` test ID, no unknown IDs, `[start, end, label]`
      triples only, integer offsets.
- [ ] `submission.zip` has `prediction.json` directly at the root (verified by `package_submission`).
- [ ] Analysis/conclusions in your report are written by you, not generated by an LLM (brief §17.5).

Phase-2 deadline per the supplied brief: **October 11, 2026, 11:59 PM**.
